# Notebook 02 : Test du Solver de CAPTCHAs

Ce notebook teste le module `solver_service.py` qui resout les CAPTCHAs avec EasyOCR.

## M2 MoSEF - Universite Paris 1 Pantheon-Sorbonne

---
## 1. Import des modules

In [ ]:
import sys
import io
import time
from pathlib import Path

# Ajouter le repertoire parent au path
sys.path.insert(0, str(Path().resolve().parent))

from app.services.captcha_generator import CaptchaGenerator
from app.services.solver_service import SolverService
import matplotlib.pyplot as plt

print("Imports reussis")

---
## 2. Initialisation des services

In [ ]:
generator = CaptchaGenerator()
solver = SolverService()

print(f"Modeles disponibles : {solver.get_available_models()}")

---
## 3. Test de resolution simple

Note : La premiere execution charge le modele EasyOCR (peut prendre 1-2 minutes).

In [ ]:
# Generer un CAPTCHA
image, true_text = generator.generate(noise_level=0.2)

# Convertir en bytes
buffer = io.BytesIO()
image.save(buffer, format="PNG")
image_bytes = buffer.getvalue()

# Resoudre
print("Resolution en cours (chargement du modele si premiere fois)...")
start = time.time()
result = solver.solve(image_bytes, model="easyocr")
elapsed = (time.time() - start) * 1000

# Resultats
predicted = result["text"]
is_correct = predicted == true_text

print(f"\nTexte reel    : {true_text}")
print(f"Texte predit  : {predicted}")
print(f"Confiance     : {result.get('confidence', 'N/A')}")
print(f"Temps         : {elapsed:.0f} ms")
print(f"Resultat      : {'CORRECT' if is_correct else 'INCORRECT'}")

# Afficher l'image
plt.figure(figsize=(8, 3))
plt.imshow(image)
plt.title(f"Reel: {true_text} | Predit: {predicted} | {'OK' if is_correct else 'ERREUR'}")
plt.axis("off")
plt.show()

---
## 4. Benchmark sur plusieurs CAPTCHAs

In [ ]:
n_tests = 10
correct = 0
total_time = 0

print(f"Test sur {n_tests} CAPTCHAs...\n")

for i in range(n_tests):
    # Generer
    image, true_text = generator.generate(noise_level=0.2)
    
    # Convertir
    buffer = io.BytesIO()
    image.save(buffer, format="PNG")
    image_bytes = buffer.getvalue()
    
    # Resoudre
    start = time.time()
    result = solver.solve(image_bytes)
    elapsed = time.time() - start
    total_time += elapsed
    
    predicted = result["text"]
    is_correct = predicted == true_text
    
    if is_correct:
        correct += 1
    
    status = "OK" if is_correct else "ERREUR"
    print(f"{i+1:2}. Reel: {true_text} | Predit: {predicted} | {status}")

# Statistiques
accuracy = correct / n_tests * 100
avg_time = total_time / n_tests * 1000

print(f"\n{'='*50}")
print(f"RESULTATS")
print(f"{'='*50}")
print(f"Precision     : {accuracy:.1f}%")
print(f"Corrects      : {correct}/{n_tests}")
print(f"Temps moyen   : {avg_time:.0f} ms")

---
## Resume

Le solver EasyOCR fonctionne correctement pour resoudre les CAPTCHAs generes.

Points cles :
- Le modele met du temps a charger la premiere fois
- Une fois charge, la resolution est rapide
- La precision depend du niveau de bruit